## Le Backtracking

Le backtracking est une technique algorithmique qui consiste à explorer différentes possibilités pour trouver une ou plusieurs solutions à un problème.

Le principe repose sur une succession de choix :

1. On effectue un choix parmi les possibilités disponibles.
2. On vérifie si ce choix respecte les contraintes du problème.
3. Si le choix est valide, on continue l'exploration.
4. Si le choix mène à une impasse, on revient en arrière (backtrack) et on essaie une autre possibilité.

L'objectif est d'explorer les possibilités sans continuer dans une branche qui ne peut pas aboutir à une solution.

### Caractéristiques

- Exploration : on construit progressivement des solutions en testant différentes possibilités.
- Rollback : on annule un choix lorsque celui-ci ne permet pas d'aboutir.
- DFS (Depth-First Search) : l'exploration se fait en profondeur.
- Implémentation : le backtracking peut être implémenté de manière récursive ou itérative. On peut directement modifier les données et effectuer un rollback en cas d'échec, ou utiliser une pile (stack) pour sauvegarder les différents états et pouvoir revenir en arrière.

### Applications

Le backtracking est notamment utilisé pour :

- Résoudre des problèmes combinatoires.
- Générer des permutations et des combinaisons.
- Résoudre des labyrinthes.
- Résoudre des problèmes de placement (N reines).
- Explorer des arbres de possibilités.

### Exemple : Sudoku

Le backtracking est une technique algorithmique qui permet de résoudre des problèmes en explorant différentes possibilités. Il consiste à modifier directement l'état actuel (par exemple, un plateau de jeu) en effectuant un choix. Si ce choix ne mène pas à une solution valide, l'algorithme annule la modification (rollback) et essaie une autre possibilité.

Ce processus se répète jusqu'à ce qu'une solution soit trouvée ou que toutes les possibilités aient été explorées.

Nous générons un plateau de Sudoku.

In [8]:

import random

def generate_sudoku(empty_cells=40):
    # Generate a valid completed Sudoku
    base = 3
    side = base * base
    def pattern(r, c):
        return (base * (r % base) + r // base + c) % side
    # Shuffle rows and columns within their groups
    def shuffle_groups():
        groups = list(range(base))
        random.shuffle(groups)
        return [
            g * base + r
            for g in groups
            for r in random.sample(range(base), base)
        ]
    rows = shuffle_groups()
    cols = shuffle_groups()
    nums = list(range(1, 10))
    random.shuffle(nums)
    # Build the completed grid
    grid = [
        [nums[pattern(r, c)] for c in cols]
        for r in rows
    ]
    # Remove random cells
    positions = [(r, c) for r in range(9) for c in range(9)]
    random.shuffle(positions)
    for r, c in positions[:empty_cells]:
        grid[r][c] = 0
    return grid

def print_sudoku(grid):
    for row in grid:
        print(" ".join(str(n) if n != 0 else "." for n in row))

# Generate a Sudoku with 40 empty cells
sudoku = generate_sudoku(50)

print_sudoku(sudoku)


. . . . 2 . 6 8 4
4 . . . 7 1 . 3 .
. . . . . . 9 . .
. 2 . . . . . 6 8
. 7 6 . . 9 . . 3
. 4 . . . . 2 . .
5 . . . . 7 . 2 .
. 3 . 8 . 4 1 . .
. 1 7 . 9 . 8 4 5


À l'aide d'une fonction qui vérifie la validité du Sudoku, nous explorons les différentes possibilités en essayant de placer les chiffres de 1 à 9 dans les cases vides. Si un chiffre ne respecte pas les règles du Sudoku, nous essayons une autre possibilité. Si aucune possibilité ne fonctionne, nous effectuons un rollback pour revenir à l'étape précédente et tester une autre solution.

In [ ]:

def is_valid(grid, row, col, num):
    # Check row
    for c in range(9):
        if grid[row][c] == num:
            return False
    # Check column
    for r in range(9):
        if grid[r][col] == num:
            return False
    # Check 3x3 square
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3
    for r in range(start_row, start_row + 3):
        for c in range(start_col, start_col + 3):
            if grid[r][c] == num:
                return False
    return True

def solve_recursive(grid):
    # Find an empty cell
    for row in range(9):
        for col in range(9):
            if grid[row][col] == 0:
                # Try numbers from 1 to 9
                for num in range(1, 10):
                    if is_valid(grid, row, col, num):
                        # Place the number
                        grid[row][col] = num
                        # Try solving the rest recursively
                        if solve_recursive(grid):
                            return True
                        # Backtrack if no solution was found
                        grid[row][col] = 0
                # No valid number works in this cell
                return False
    # No empty cells remain: Sudoku is solved
    return True

def solve_iterative(grid):
    # Store the empty cells
    empty_cells = []
    for row in range(9):
        for col in range(9):
            if grid[row][col] == 0:
                empty_cells.append((row, col))
    # Keep track of the current cell and number to try
    index = 0
    next_num = [1] * len(empty_cells)
    while index < len(empty_cells):
        row, col = empty_cells[index]
        found = False
        # Try numbers from where we left off
        for num in range(next_num[index], 10):
            if is_valid(grid, row, col, num):
                grid[row][col] = num
                # Move to the next cell
                next_num[index] = num + 1
                index += 1
                # Reset the next cell's starting number
                if index < len(empty_cells):
                    next_num[index] = 1
                found = True
                break
        if not found:
            # No number works: reset this cell
            grid[row][col] = 0
            next_num[index] = 1
            # Go back to the previous cell
            index -= 1
            # If we went before the first cell, no solution exists
            if index < 0:
                return False
    # All empty cells have been filled
    return True

solve_iterative(sudoku)

print_sudoku(sudoku)

7 5 1 9 2 3 6 8 4
4 9 8 6 7 1 5 3 2
2 6 3 5 4 8 9 1 7
1 2 9 4 3 5 7 6 8
8 7 6 2 1 9 4 5 3
3 4 5 7 8 6 2 9 1
5 8 4 1 6 7 3 2 9
9 3 2 8 5 4 1 7 6
6 1 7 3 9 2 8 4 5


## Problème des N reines (N-Queens)

Voici un exemple de code utilisé pour résoudre le problème des N reines avec la méthode du *backtracking* (retour sur trace).

Le problème des N reines consiste à placer N reines sur un plateau de N×N cases de sorte qu'aucune reine ne puisse en attaquer une autre. Autrement dit, deux reines ne doivent jamais se trouver sur la même ligne, la même colonne ou la même diagonale.

In [10]:
# Exemple avec le problème N-Queens

# Dans cet exercice, il existe différentes façons de mémoriser l'endroit où l'on place une reine sur le plateau.
# Ici, nous allons utiliser un tableau de N éléments où l'index i représente la ligne et la valeur la colonne de la reine.
# On peut se permettre de faire cela car il ne peut y avoir qu'une reine par ligne ; -1 indiquera qu'aucune reine
# n'a été placée pour l'instant sur cette ligne.

# On peut nous demander plusieurs choses : soit renvoyer toutes les possibilités, soit une seule.
# Ici, nous renverrons la première trouvée.

def n_queens(n):
    # Création d'une mémoire nous permettant de revenir en arrière dans le cas où l'on est bloqué.
    # Chaque élément de la mémoire est un état complet, composé de :
    #   - le plateau (tableau de N éléments décrit plus haut)
    #   - l'ensemble des colonnes déjà occupées
    #   - l'ensemble des diagonales "ligne - colonne" déjà occupées (diagonales descendantes)
    #   - l'ensemble des diagonales "ligne + colonne" déjà occupées (diagonales montantes)
    # Comme chaque état transporte ses propres ensembles, on n'a jamais besoin de retirer quoi que ce soit
    # lorsqu'on revient en arrière : un état abandonné emporte simplement ses ensembles avec lui.
    # On utilise des frozenset (ensembles immuables) pour être sûr de ne jamais modifier par erreur
    # les ensembles d'un autre état : on ne fait que créer de nouvelles versions.
    mem = [([-1]*n, frozenset(), frozenset(), frozenset())]

    while(mem):
        # mem est utilisée comme une pile : pop() reprend le dernier état empilé,
        # ce qui revient à explorer en profondeur puis à revenir en arrière en cas d'impasse
        actuel, set_col, set_diag_1, set_diag_2 = mem.pop()
        print(actuel)
        # Nous prenons le premier index égal à -1, donc la ligne où nous allons ajouter notre prochaine reine
        premier_index = next(i for i, v in enumerate(actuel) if v == -1)
        # Nous essayons tous les cas possibles ; si nous avons une solution, nous la renvoyons
        for x in range(n) :
            # Vérification des conditions liées à la colonne et aux diagonales.
            # Pas besoin de vérifier la condition liée à la ligne, car la manière dont on garde les positions
            # le fait naturellement (une reine par ligne).
            # Deux reines sont sur la même diagonale si elles ont la même valeur de (ligne - colonne)
            # ou la même valeur de (ligne + colonne).
            if not x in set_col and not premier_index - x in set_diag_1 and not premier_index + x in set_diag_2:
                temp = actuel.copy()
                temp[premier_index] = x
                if premier_index == n-1 :
                    # Derniere ligne placée et la position est correcte donc on a une solution
                    return temp
                # La position est valide mais incomplète : on l'empile avec ses nouveaux ensembles.
                # L'opérateur | crée un nouvel ensemble, ceux de l'état actuel restent donc intacts
                # et pourront être réutilisés pour les autres colonnes de cette ligne.
                mem.append((temp, set_col | {x}, set_diag_1 | {premier_index - x}, set_diag_2 | {premier_index + x}))
    # Aucune solution pour N
    return -1

In [11]:
print(n_queens(4))

[-1, -1, -1, -1]
[3, -1, -1, -1]
[3, 1, -1, -1]
[3, 0, -1, -1]
[3, 0, 2, -1]
[2, -1, -1, -1]
[2, 0, -1, -1]
[2, 0, 3, -1]
[2, 0, 3, 1]


# Déroulé de l'algorithme pour N = 4

**Légende** : 👑 reine placée · ❌ essai refusé (la reine serait attaquée) · ⬜ case vide

À chaque étape, on **dépile** un état (le dernier empilé), puis on essaie de placer une reine
dans chaque colonne de la ligne suivante. Les essais valides sont empilés, les autres sont abandonnés.

---

## Étape 1 : état initial `[-1, -1, -1, -1]`

On essaie les 4 colonnes de la ligne 0. Aucune reine n'est encore placée, donc tout est valide.

| x = 0 ✅ | x = 1 ✅ | x = 2 ✅ | x = 3 ✅ |
|:---:|:---:|:---:|:---:|
| 👑⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜👑⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ |

Les 4 états sont empilés. Le dernier empilé est `[3, ...]`, c'est lui qu'on explore en premier.

---

## Étape 2 : on dépile `[3, -1, -1, -1]`

On essaie les colonnes de la ligne 1.

| x = 0 ✅ | x = 1 ✅ | x = 2 ❌ | x = 3 ❌ |
|:---:|:---:|:---:|:---:|
| ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜👑⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜⬜❌⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜⬜⬜❌<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ |
| | | diagonale | même colonne |

On empile `[3, 0]` puis `[3, 1]`.

---

## Étape 3 : on dépile `[3, 1, -1, -1]`

On essaie les colonnes de la ligne 2.

| x = 0 ❌ | x = 1 ❌ | x = 2 ❌ | x = 3 ❌ |
|:---:|:---:|:---:|:---:|
| ⬜⬜⬜👑<br>⬜👑⬜⬜<br>❌⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜👑⬜⬜<br>⬜❌⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜👑⬜⬜<br>⬜⬜❌⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>⬜👑⬜⬜<br>⬜⬜⬜❌<br>⬜⬜⬜⬜ |
| diagonale | même colonne | diagonale | même colonne |

🔙 **Impasse** : rien n'est empilé. On revient en arrière en dépilant l'état suivant.

---

## Étape 4 : on dépile `[3, 0, -1, -1]`

On essaie les colonnes de la ligne 2.

| x = 0 ❌ | x = 1 ❌ | x = 2 ✅ | x = 3 ❌ |
|:---:|:---:|:---:|:---:|
| ⬜⬜⬜👑<br>👑⬜⬜⬜<br>❌⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜❌⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜👑⬜<br>⬜⬜⬜⬜ | ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜⬜❌<br>⬜⬜⬜⬜ |
| même colonne | diagonale | | même colonne |

On empile `[3, 0, 2]`.

---

## Étape 5 : on dépile `[3, 0, 2, -1]`

On essaie les colonnes de la ligne 3.

| x = 0 ❌ | x = 1 ❌ | x = 2 ❌ | x = 3 ❌ |
|:---:|:---:|:---:|:---:|
| ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜👑⬜<br>❌⬜⬜⬜ | ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜👑⬜<br>⬜❌⬜⬜ | ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜👑⬜<br>⬜⬜❌⬜ | ⬜⬜⬜👑<br>👑⬜⬜⬜<br>⬜⬜👑⬜<br>⬜⬜⬜❌ |
| même colonne | diagonale | même colonne | même colonne |

🔙 **Impasse** : toute la branche `[3, ...]` est épuisée. La pile ne contient plus que `[0]`, `[1]`, `[2]`.
On revient donc au tout début, à la ligne 0, et on essaie la colonne 2.

---

## Étape 6 : on dépile `[2, -1, -1, -1]`

On essaie les colonnes de la ligne 1.

| x = 0 ✅ | x = 1 ❌ | x = 2 ❌ | x = 3 ❌ |
|:---:|:---:|:---:|:---:|
| ⬜⬜👑⬜<br>👑⬜⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>⬜❌⬜⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>⬜⬜❌⬜<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>⬜⬜⬜❌<br>⬜⬜⬜⬜<br>⬜⬜⬜⬜ |
| | diagonale | même colonne | diagonale |

On empile `[2, 0]`.

---

## Étape 7 : on dépile `[2, 0, -1, -1]`

On essaie les colonnes de la ligne 2.

| x = 0 ❌ | x = 1 ❌ | x = 2 ❌ | x = 3 ✅ |
|:---:|:---:|:---:|:---:|
| ⬜⬜👑⬜<br>👑⬜⬜⬜<br>❌⬜⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>👑⬜⬜⬜<br>⬜❌⬜⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>👑⬜⬜⬜<br>⬜⬜❌⬜<br>⬜⬜⬜⬜ | ⬜⬜👑⬜<br>👑⬜⬜⬜<br>⬜⬜⬜👑<br>⬜⬜⬜⬜ |
| même colonne | diagonale | même colonne | |

On empile `[2, 0, 3]`.

---

## Étape 8 : on dépile `[2, 0, 3, -1]`

On essaie les colonnes de la dernière ligne.

| x = 0 ❌ | x = 1 🎉 SOLUTION |
|:---:|:---:|
| ⬜⬜👑⬜<br>👑⬜⬜⬜<br>⬜⬜⬜👑<br>❌⬜⬜⬜ | ⬜⬜👑⬜<br>👑⬜⬜⬜<br>⬜⬜⬜👑<br>⬜👑⬜⬜ |
| même colonne | dernière ligne placée et valide |

L'algorithme s'arrête ici et renvoie `[2, 0, 3, 1]`. Les colonnes 2 et 3 de la dernière ligne ne sont même pas testées.

---

# Vue d'ensemble de l'arbre

```
[-1,-1,-1,-1]
├── [0,...]          (jamais exploré : solution trouvée avant)
├── [1,...]          (jamais exploré : solution trouvée avant)
├── [2,...]          étape 6
│   ├── [2,0]        étape 7
│   │   ├── [2,0,0] ❌   [2,0,1] ❌   [2,0,2] ❌
│   │   └── [2,0,3]  étape 8
│   │       ├── [2,0,3,0] ❌
│   │       └── [2,0,3,1] 🎉 SOLUTION
│   └── [2,1] ❌   [2,2] ❌   [2,3] ❌
└── [3,...]          étape 2
    ├── [3,0]        étape 4  →  [3,0,2] (étape 5) → impasse
    ├── [3,1]        étape 3  →  impasse
    └── [3,2] ❌   [3,3] ❌
```

💡 **À retenir** : le backtracking, c'est ce qui se passe entre les étapes 5 et 6. Quand une branche est
une impasse (étapes 3 et 5), on ne recommence pas de zéro : on dépile simplement le dernier choix
laissé en attente et on repart de là.